# Logistics Data

This notebook performs comprehensive data quality checks on the Logistics_Data dataset.


## 1. Data Loading


In [2]:
import pandas as pd

# Load the Logistics_Data data
df = pd.read_csv('../datasets/Logistics_Data.csv')

print("Data loaded successfully!")
print(f"Total records: {len(df)}")


Data loaded successfully!
Total records: 150


## 2. Basic Data Exploration


In [3]:
# Display first few rows
df.head()


,Shipment ID,Site ID,Product ID,Shipment Date,Quantity,Delivery Status,Transportation Type
0,SHP90000,7EEQ,PRD10038,2024-07-15,76,Cancelled,Rail
1,SHP90001,NY20,PRD10050,2024-08-30,92,Delivered,Rail
2,SHP90002,0L72,PRD10049,2024-05-28,40,Delayed,Truck
3,SHP90003,58KX,PRD10001,2024-10-02,50,Delivered,Ship
4,SHP90004,4ZKL,PRD10053,2024-10-05,79,Cancelled,Rail


In [4]:
# Display column names
df.columns


Index(['Shipment ID', 'Site ID', 'Product ID', 'Shipment Date', 'Quantity',
       'Delivery Status', 'Transportation Type'],
      dtype='object')

In [5]:
# Display dataset shape
df.shape


(150, 7)

In [6]:
# Display data types and basic info
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Shipment ID          150 non-null    object
 1   Site ID              150 non-null    object
 2   Product ID           150 non-null    object
 3   Shipment Date        150 non-null    object
 4   Quantity             150 non-null    int64 
 5   Delivery Status      150 non-null    object
 6   Transportation Type  150 non-null    object
dtypes: int64(1), object(6)
memory usage: 8.3+ KB


## 3. Data Quality Checks

### 3.1 Missing Values Analysis


In [7]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
})

print("=== Missing Values Analysis ===")
print(missing_df)
print(f"\nTotal missing values: {df.isnull().sum().sum()}")


=== Missing Values Analysis ===
                     Missing Values  Percentage
Shipment ID                       0         0.0
Site ID                           0         0.0
Product ID                        0         0.0
Shipment Date                     0         0.0
Quantity                          0         0.0
Delivery Status                   0         0.0
Transportation Type               0         0.0

Total missing values: 0


### 3.2 Duplicate Records Check


In [8]:
# Check for duplicate records
duplicates_all = df.duplicated().sum()
print("=== Duplicate Records Check ===")
print(f"Total duplicate rows: {duplicates_all}")


=== Duplicate Records Check ===
Total duplicate rows: 0


### 3.3 Data Type Validation


In [9]:
# Check data types
print("=== Data Type Validation ===")
print(df.dtypes)

# Verify expected data types
expected_types = {
    'Shipment ID': 'object',
    'Site ID': 'object',
    'Product ID': 'object',
    'Shipment Date': 'object',
    'Quantity': 'int64',
    'Delivery Status': 'object',
    'Transportation Type': 'object'
}

type_issues = []
for col, expected in expected_types.items():
    if col in df.columns and df[col].dtype != expected:
        type_issues.append(f"{col}: expected {expected}, got {df[col].dtype}")

if type_issues:
    print("\nData Type Issues:")
    for issue in type_issues:
        print(f"  - {issue}")
else:
    print("\nAll data types are correct!")


=== Data Type Validation ===
Shipment ID            object
Site ID                object
Product ID             object
Shipment Date          object
Quantity                int64
Delivery Status        object
Transportation Type    object
dtype: object

All data types are correct!


### 3.4 Value Range Checks


In [10]:
# Check value ranges for numeric columns
print("=== Value Range Checks ===")

# Get numeric columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numeric_cols:
    min_val = df[col].min()
    max_val = df[col].max()
    print(f"\n{col}: Min={min_val}, Max={max_val}")
    
    # Check for negative values (except where appropriate)
    if 'ID' not in col and 'Flag' not in col:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            print(f"  Negative values found: {neg_count}")


=== Value Range Checks ===

Quantity: Min=10, Max=100


### 3.5 Categorical Value Validation


In [11]:
# Check unique values in categorical columns
print("=== Categorical Value Validation ===")

cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    if 'ID' not in col and 'Date' not in col and 'Name' not in col:
        unique_vals = df[col].unique()
        print(f"\n{col} unique values: {unique_vals}")


=== Categorical Value Validation ===

Delivery Status unique values: ['Cancelled' 'Delivered' 'Delayed']

Transportation Type unique values: ['Rail' 'Truck' 'Ship' 'Air']


### 3.6 Outlier Detection


In [12]:
# Detect outliers using IQR method
print("=== Outlier Detection (IQR Method) ===")

def detect_outliers_iqr(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

# Check outliers for numeric columns
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_columns:
    if 'ID' not in col:
        count, lower, upper = detect_outliers_iqr(col)
        print(f"\n{col}:")
        print(f"  Outliers: {count}")
        print(f"  Lower Bound: {lower:.2f}")
        print(f"  Upper Bound: {upper:.2f}")


=== Outlier Detection (IQR Method) ===

Quantity:
  Outliers: 0
  Lower Bound: -40.88
  Upper Bound: 146.12


## 4. Data Quality Summary


In [13]:
# Generate comprehensive data quality summary
print("=" * 50)
print("DATA QUALITY SUMMARY")
print("=" * 50)

print(f"\n1. Dataset Overview:")
print(f"   - Total Records: {len(df)}")
print(f"   - Total Columns: {len(df.columns)}")

print(f"\n2. Missing Values:")
print(f"   - Total Missing: {df.isnull().sum().sum()}")
for col in df.columns:
    missing = df[col].isnull().sum()
    print(f"   - {col}: {missing}")

print(f"\n3. Duplicate Records:")
print(f"   - Duplicate Rows: {df.duplicated().sum()}")

print(f"\n4. Data Quality Status:")
quality_passed = True

if df.isnull().sum().sum() > 0:
    print(f"   ❌ Missing values found")
    quality_passed = False
else:
    print(f"   ✓ No missing values")

if df.duplicated().sum() > 0:
    print(f"   ❌ Duplicate records found")
    quality_passed = False
else:
    print(f"   ✓ No duplicate records")

if quality_passed:
    print(f"\n✅ DATA QUALITY CHECK PASSED!")
else:
    print(f"\n❌ DATA QUALITY ISSUES DETECTED - REVIEW REQUIRED!")


DATA QUALITY SUMMARY

1. Dataset Overview:
   - Total Records: 150
   - Total Columns: 7

2. Missing Values:
   - Total Missing: 0
   - Shipment ID: 0
   - Site ID: 0
   - Product ID: 0
   - Shipment Date: 0
   - Quantity: 0
   - Delivery Status: 0
   - Transportation Type: 0

3. Duplicate Records:
   - Duplicate Rows: 0

4. Data Quality Status:
   ✓ No missing values
   ✓ No duplicate records

✅ DATA QUALITY CHECK PASSED!
